Install the required packages

In [ ]:
pip install langchain langchain-openai langchain-community langchain-core chromadb tiktoken pypdf python-dotenv   

### Import the libraries


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from openai import OpenAI

In [8]:
def pdf_chunker(pdf_file):
    complete_chunks = []

    pdf_loader = PyPDFLoader(file_path= pdf_file)
    pdf_pages = pdf_loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap= 50)
    
    for pages in pdf_pages:
        complete_chunks.extend(text_splitter.split_text(pages.page_content))

    return complete_chunks

complete_chunks = pdf_chunker(pdf_file = r"D:\Data Science_bat\Gen_ai\war_document.pdf")
print(f"Chunking completed: {complete_chunks[0][:100]}.......")

Chunking completed: World War I
drishtiias.com
/printpdf/world-war-i-1
World War I (WW I), also known as the Great War, .......


In [ ]:
# you can skip if data is already in database

def get_embeddings_for_all_chunks(complete_chunks):
    all_documents_to_insert = []

    for chunk in complete_chunks:
        document = {}
        document['text'] = chunk
        document['embeddings'] = get_embeddings(chunk)
        all_documents_to_insert.append(document)

    return all_documents_to_insert

embeddings_for_mongodb = get_embeddings_for_all_chunks(complete_chunks)

In [ ]:
uri = "mongodb+srv://  :  @ragappcluster.p8mndqc.mongodb.net/?appName=  "

# Create a new client and connect to the server
client = MongoClient(uri, server_api=ServerApi('1'))

# connect to the database and particular table[collection]
collections = client["rag_new_db"]["war_documents"]

In [ ]:
# you can skip if data is already in database
collections.insert_many(embeddings_for_mongodb)

Create Vector Search Index for finding similar documents when user asks the question 

Excecute only once

In [ ]:
# you can skip if data is already in database and indexing is completed

from pymongo.operations import SearchIndexModel
import time

index_name = "rag_index"
# Name for our search index — we'll reference this name when querying

search_index_model = SearchIndexModel(
  definition={
    "fields": [
      {
        "type": "vector",          # This field contains vector data
        "numDimensions": 1024,     # Each vector has 2048 numbers (must match our model)
        "path": "embeddings",       # The field name in our documents that holds the vector
        "similarity": "cosine"     # Use cosine similarity to compare vectors
        # Cosine similarity measures the angle between two vectors
        # Closer to 1.0 = more similar meaning
      }
    ]
  },
  name=index_name,
  type="vectorSearch"              # This is a vector search index (not a regular text index)
)

collections.create_search_index(model=search_index_model)
# Tell MongoDB Atlas to build the vector search index
# This enables fast similarity search over our embeddings

print("Polling to check if the index is ready. This may take up to a minute.")
predicate = None
if predicate is None:
   predicate = lambda index: index.get("queryable") is True
   # A small function that checks if the index is ready to accept queries

while True:
   indices = list(collections.list_search_indexes(index_name))
   # Ask MongoDB for the current status of our index
   if len(indices) and predicate(indices[0]):
      break
      # If the index exists and is queryable, stop waiting
   time.sleep(5)
   # Otherwise wait 5 seconds and check again

print(index_name + " is ready for querying.")

In [ ]:
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=" ",
)

def get_embeddings(chunk_text):

    # Image input embeddings use multimodal content format
    embedding = client.embeddings.create(
    model="nvidia/llama-nemotron-embed-vl-1b-v2:free",
    dimensions= 1024,
    input=[
        {
        "content": [
            {"type": "text", "text": chunk_text}
        ]
        }
    ],
    encoding_format="float"
    )

    return embedding.data[0].embedding


In [25]:
def get_relevent_doc_from_db(query):
    query_embedding = get_embeddings(query)
    # Convert our search query into a vector (same way we did for documents)

    results = collections.aggregate([
        {
            "$vectorSearch": {
            "index": "rag_index",
            "queryVector": query_embedding,
            "path": "embeddings",
            "numCandidates": 1024,
            "limit": 5
            # Step 2: Find the 5 most similar chunks in MongoDB
            }
        },
        {
            "$project": {
            "_id": 0,      # Exclude the MongoDB ID from results
            "text": 1      # Include only the "text" field
            # Step 3: Return just the text, not the embedding vectors
            }
        }
    ])

    array_of_results = []
    for doc in results:
        array_of_results.append(doc['text'])

    return " ".join(array_of_results)




In [26]:
def chat_with_model(Query, user_query_related_text):

    prompt = f"""Use the following pieces of context to answer the question at the end.
    {user_query_related_text}
    Question: {Query}"""

    resonse = client.chat.completions.create(
    model = "openai/gpt-oss-120b:free",
    messages=[
        {
            'role':'user',
            "content": prompt
        }
    ]
    )
    print(resonse.choices[0].message.content)


Query = "what is this war is all about ? explain in 2-3 sentence"
related_docs = get_relevent_doc_from_db(query=Query)
chat_with_model(Query , related_docs)

World War I (1914‑1918), often called the Great War, was a global conflict that pitted the Allied Powers—initially France, Russia and Britain (later joined by the United States)—against the Central Powers led by Germany, Austria‑Hungary and the Ottoman Empire.  It was sparked by a mixture of nationalist rivalries, imperial competition, militaristic arms races, and the immediate catalyst of the assassination of Archduke Franz Ferdinand, and it resulted in massive military casualties, political upheavals and economic disruption worldwide.


World War I (1914‑1918), often called the Great War, was a global conflict that pitted the Allied Powers—initially France, Russia and Britain (later joined by the United States)—against the Central Powers led by Germany, Austria‑Hungary and the Ottoman Empire.  It was sparked by a mixture of nationalist rivalries, imperial competition, militaristic arms races, and the immediate catalyst of the assassination of Archduke Franz Ferdinand, and it resulted in massive military casualties, political upheavals and economic disruption worldwide.

SyntaxError: invalid non-printable character U+202F (3626476016.py, line 1)